# 2. From executed prices to historical mispricing

Becker's mispricing is **mean realized payout minus mean purchase-implied probability** over a
set of positions. This notebook explains that definition, the audited CSV aggregation and the
equal-cost YES/NO comparison. Descriptive historical differences are not yet predictive findings.
The older mispricing/structural notebooks remain useful detailed interfaces to the exploratory
heterogeneity analyses; here we connect them to the final supervised experiment.

## 2.1 Setup and explicit input selection

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "extension/clustering/supervised/pipeline.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside the prediction-market-analysis repository.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "extension/notebooks/results/.matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(ROOT / "extension/notebooks/results/.cache"))

In [ ]:
import json

import pandas as pd
from IPython.display import Markdown, display
from threadpoolctl import threadpool_limits

from extension.notebooks.support import (
    begin_stage,
    finish_stage,
    output_dir,
    published_paths,
    read_completed,
)

published, DATA, ASSIGNMENTS, STATS, RESULTS = published_paths(ROOT)
print("Completed experiment:", RESULTS.relative_to(ROOT))
print("CSV split design:", published.get("split", "See methodology"))

In [ ]:
RUN_NAME = "completion_walkthrough"
RECOMPUTE = False  # Explicit opt-in: expensive steps write isolated notebook artifacts.
USE_REBUILT_UPSTREAM = False  # True only after completing the preceding rebuild notebooks.

measurement_output = output_dir(ROOT, RUN_NAME, "measurement")
if USE_REBUILT_UPSTREAM:
    rebuilt_dir = output_dir(ROOT, RUN_NAME, "clustering")
    read_completed(rebuilt_dir, stage="clustering")
    assignment_input = rebuilt_dir / "assignments.csv"
    if not RECOMPUTE:
        raise ValueError("New labels require RECOMPUTE=True and fresh CSV extraction.")
else:
    assignment_input = ASSIGNMENTS

## 2.2 Outcomes are labels, not predictors
At YES price p, the observed deviation is `y - p/100`; the complementary NO position costs
`100-p` and has outcome `1-y`. A 30-cent YES contract settling YES has deviation +70 points;
settling NO has deviation -30 points. Neither single observation proves mispricing.
This tiny **synthetic arithmetic example** explains units only; it is never used for findings.

In [ ]:
from extension.clustering.mispricing.statistics import curve, estimate, positions

arithmetic = pd.DataFrame(
    {
        "ticker": ["illustration"],
        "event_id": ["illustration"],
        "cluster": ["illustration"],
        "split": ["test"],
        "p": [30],
        "outcome": [1],
        "n": [1],
    }
)
display(curve(positions(arithmetic, "n")))

## 2.3 Audit the real CSV extract
The cache counts executions by contract and exact YES price, retaining execution count n and
traded units separately. It excludes invalid/noncomplementary prices and invalid trades, requires
resolved binary outcomes and at least $100 observed-window notional, and audits missing/duplicate
trade IDs. Every execution is kept within the extract, not a fixed 72-hour snapshot.
`sum(count)` is pair notional because complementary positions together cost $1 per unit; it is
not final lifetime market volume. Final status and full-window eligibility are retrospective screens.
Review mode verifies cached input metadata against the actual files before reading summaries.

In [ ]:
from extension.clustering.mispricing.data import extract
from extension.clustering.mispricing.pipeline import fingerprint

if RECOMPUTE:
    run = begin_stage(
        ROOT,
        measurement_output,
        "measurement",
        [DATA / "master/markets.csv", DATA / "master/trades.csv", assignment_input],
        {"weighting": "execution primary; unit sensitivity", "assignments": str(assignment_input)},
    )
    cells, audit = extract(DATA, assignment_input, measurement_output)
    audit["inputs"] = fingerprint([DATA / "master/markets.csv", DATA / "master/trades.csv", assignment_input])
    (measurement_output / "data_audit.json").write_text(json.dumps(audit, indent=2) + "\n")
    statistics_input = measurement_output / "sufficient_statistics.csv.gz"
else:
    statistics_input = STATS
    audit = json.loads((STATS.parent / "data_audit.json").read_text())
    assert audit["inputs"] == fingerprint([DATA / "master/markets.csv", DATA / "master/trades.csv", ASSIGNMENTS])
    read_completed(STATS.parent)
display(pd.Series({k: v for k, v in audit.items() if k != "inputs"}, name="CSV audit"))

## 2.4 Inspect sufficient statistics and compute an exact-cost example
Each row retains a contract's one settlement outcome but can summarize many executions. We
weight means by n, not by treating the row or its repeated trades as independent events.
The following **bounded leading cache sample** illustrates the computation only. Its means
are not the full-population estimates or inputs to the paper's conclusions.

In [ ]:
sample = pd.read_csv(statistics_input, keep_default_na=False, nrows=2000)
display(sample[["ticker", "event_id", "p", "outcome", "n", "units", "split", "cluster"]].head(10))
sample_positions = positions(sample, "n")
sample_curves = curve(sample_positions)
sample_curves["calibration_error_pp"] = 100 * sample_curves.win_rate - sample_curves.cost
display(sample_curves.head(12))

## 2.5 Match purchase costs, not opposing sides of one trade
YES at 10 cents must be compared with NO at 10 cents, which comes from a YES execution at
90 cents. Comparing the two complementary positions of the same execution would compare
10 cents with 90 cents and answer a different question. The full exact-cost table below is
from the completed descriptive run; it is not a new independent replication.

In [ ]:
if not RECOMPUTE:
    descriptive_curves = pd.read_csv(STATS.parent / "pooled_curves.csv")
    display(descriptive_curves.head(14))
else:
    descriptive_curves = []
    populations = {
        "all_resolved": cells,
        "non_combination": cells[cells.non_combination.eq(True)],
        "assigned_matched": cells[cells.cluster.ne("")],
    }
    for population, subset in populations.items():
        for weighting, column in [("execution", "n"), ("unit", "units")]:
            result = curve(positions(subset, column))
            result["population"], result["weighting"] = population, weighting
            descriptive_curves.append(result)
    descriptive_curves = pd.concat(descriptive_curves, ignore_index=True)
    descriptive_curves.to_csv(measurement_output / "pooled_curves.csv", index=False)
    display(descriptive_curves.head(14))

## 2.6 Historical heterogeneity and structural-only groups
The original descriptive workflow pools splits and asks whether group wedges differ; that is
not an out-of-sample prediction test. Its inference uses event/market blocks, price support
screens and multiplicity checks. Rule-only groups provide a separate structural comparison.
These original exploratory results retain their original eligibility rules: the final predictive
cohort later excludes explicit multi-award combinations and imposes stricter training support.
Do not silently substitute descriptive pooled-split outcomes for training-only baseline estimates.

In [ ]:
if RECOMPUTE:
    summaries = []
    tests = []
    assigned_cells = cells[cells.cluster.ne("")]
    with threadpool_limits(limits=2):
        for weighting, column in [("execution", "n"), ("unit", "units")]:
            pos = positions(assigned_cells, column)
            for reference in ["pooled", "family"]:
                rows, test = estimate(pos, (1, 49), 10, 20, 10, weighting, reference=reference)
                summaries.append(rows)
                tests.append({"weighting": weighting, "reference": reference, **test})
    pd.concat(summaries, ignore_index=True).to_csv(measurement_output / "cluster_summary.csv", index=False)
    pd.DataFrame(tests).to_csv(measurement_output / "heterogeneity_tests.csv", index=False)
    finish_stage(
        measurement_output,
        run,
        audit=audit,
        note="Cache and overall cluster contrasts rebuilt; not the complete old band/structural report.",
    )
else:
    for name in ["cluster_summary.csv", "heterogeneity_tests.csv"]:
        display(Markdown("**Recorded " + name + "**"))
        display(pd.read_csv(STATS.parent / name).head(10))

In [ ]:
from extension.clustering.mispricing.structural import structural_labels

market_example = pd.read_csv(
    DATA / "master/markets.csv",
    nrows=2000,
    keep_default_na=False,
    usecols=["ticker", "event_ticker", "market_type", "title", "yes_sub_title"],
)
display(structural_labels(market_example).head(8))
print("Bounded rule illustration; full structural results are documented in STRUCTURAL_FINDINGS.md.")

## 2.7 What changes for prediction?
Descriptive differences motivate a hypothesis; they do not show that group biases persist.
The final experiment fits on training outcomes only, selects settings using validation, and
reports test scores. It compares all methods on exactly the same supported positions.
The next notebook adds the historical-transfer baselines and supervised models.

References: [descriptive findings](../clustering/mispricing/FINDINGS.md),
[rule-only findings](../clustering/mispricing/STRUCTURAL_FINDINGS.md),
[prior full descriptive notebook](../../notebooks/extension_prototype/03_csv_mispricing.ipynb),
[prior full structural notebook](../../notebooks/extension_prototype/04_structural_families.ipynb).